<a href="https://colab.research.google.com/github/rpizarrog/machine_learning_r_python_casos_de_estudio/blob/main/notebook_Python/Caso_ejemplo_general_con_datos_de_comportamienos_de_compradores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Contexto

Se trata de información de comportamiento de compras de los clientes en un modelo híbrido desde los que lo hace a través de comercio electrónico como de los que tradicionalmente lo hacen acudiendo a la tienda.

Este conjunto de datos simula el comportamiento del consumidor en el mundo real combinando información del tipo demográfico, hábitos digitales, patrones de compra, preferencias logísticas y factores psicológicos.

Es un conjunto de datos estructurado de 11789 registros y 25 columnas que pueden ser analizadas mediante posibles modelos de clasificación del aprendizaje automático.

Se sugiere incluir los atributos del conjunto de datos, por ejemplo:

* La edad del cliente en años cumplidos,
* Los ingresos mensuales,
* Horas de uso diarias en internet u horas en línea por día,
* Años con el uso con su teléfono celular ,
* Horas diarias en redes sociales,
* Valor numérico de la confianza del cliente en hacer sus pagos,
* Puntuación de nivel de comodidad con el uso de la tecnología refiriéndose al uso de aplicaciones en línea,
* Número de pedidos en línea por mes,
* Número de visitas a tienda por mes,
* Promedio de compras en línea,
* Promedio de visitas al mes a la tienda, entre otros…


# Objetivo

Implementar y evaluar un modelo de clasificación por medio del algoritmo regresión logística multinomial
para predecir el comportamiento de compra de los clientes: si lo hará en la tienda, por internet en línea o ambas.

# Descripción

Dataset: "Online vs In-store Shopping Behaviour Dataset"
Fuente Kaggle:
https://www.kaggle.com/datasets/shree0910/online-vs-in-store-shopping-behaviour-dataset

Copia en GitHub:
https://github.com/rpizarrog/machine_learning_r_python_casos_de_estudio/blob/main/datos/online%20vs%20store%20shopping%20dataset.csv

Contexto: simula comportamiento del consumidor (demografía, hábitos digitales, patrones de compra,
preferencias logísticas y factores psicológicos).

Estructura: 11,789 registros y 25 columnas.
Variable dependiente: shopping_preference (Hybrid, Online, Store).

## Cargar librerías

In [1]:
import numpy as np  # Para tratar con estructuras de datos
import pandas as pd # Para cargar datos y procesar conjunto de datos

import statsmodels.api as sm # Para modelos de regresión logística
import statsmodels.formula.api as smf # Por construir modelos de regresión logística

from sklearn.model_selection import train_test_split # Para particionar datos
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report # Para evaluar modelos

from sklearn.preprocessing import StandardScaler # Para escalar datos

from sklearn.metrics import confusion_matrix # Para evaluar modelos de clasificación con matriz de confusión


## Cargar funciones

Se declaran las funciones de este caso



In [2]:
def f_cargar_datos(ruta_archivo: str) -> pd.DataFrame:
    """
    f_cargar_datos()
    Objetivo:
      Importar datos desde un archivo CSV (URL o archivo local).
    Retorna:
      DataFrame listo para análisis.
    """
    return pd.read_csv(ruta_archivo)


def f_visualizar_head_tail_reducido(datos: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    """
    f_visualizar_head_tail_reducido()
    Objetivo:
      Mostrar primeros n y últimos n registros en una misma tabla,
      visualizando únicamente:
        - Los primeros 4 atributos
        - Los últimos 3 atributos
      Insertando una fila con "..." como separador.

    Nota:
      Se convierte a string SOLO para visualización, para evitar choques de tipos.

    Retorna:
      DataFrame (para mostrar en notebook).
    """
    total_columnas = datos.shape[1]
    idx_prim = list(range(0, min(4, total_columnas)))
    idx_ult = list(range(max(total_columnas - 3, 0), total_columnas))
    idx = sorted(set(idx_prim + idx_ult))

    datos_red = datos.iloc[:, idx].copy()
    head_df = datos_red.head(n).astype(str)
    tail_df = datos_red.tail(n).astype(str)

    fila_puntos = pd.DataFrame([["..."] * datos_red.shape[1]], columns=datos_red.columns)
    tabla_final = pd.concat([head_df, fila_puntos, tail_df], ignore_index=True)
    return tabla_final


def f_describir_datos(datos: pd.DataFrame) -> dict:
    """
    f_describir_datos()
    Objetivo:
      Generar estadísticas descriptivas básicas y estructura.
    Retorna:
      dict con:
        - summary (describe)
        - structure (texto tipo info)
    """
    import io
    s = io.StringIO()
    datos.info(buf=s)
    structure = s.getvalue()

    summary = datos.describe(include="all")
    return {"summary": summary, "structure": structure}


def f_transformar_factor(datos: pd.DataFrame) -> pd.DataFrame:
    """
    f_transformar_factor()
    Objetivo:
      Convertir variables tipo object (texto) a tipo category (similar a factor en R).
    Retorna:
      DataFrame con conversiones aplicadas.
    """
    datos_out = datos.copy()
    cols_obj = datos_out.select_dtypes(include=["object"]).columns
    for c in cols_obj:
        datos_out[c] = datos_out[c].astype("category")
    return datos_out


def f_balancear_clases_undersamplig(datos: pd.DataFrame, variable_objetivo: str) -> pd.DataFrame:
    """
    f_balancear_clases_undersamplig()
    Objetivo:
      Balancear clases por undersampling a tamaño de la clase minoritaria.
    Nota:
      Reduce tamaño del dataset (pérdida de información).
    """
    np.random.seed(2026)

    conteos = datos[variable_objetivo].value_counts()
    min_n = int(conteos.min())

    partes = []
    for clase, dfc in datos.groupby(variable_objetivo):
        partes.append(dfc.sample(n=min_n, replace=False, random_state=2026))

    datos_balanceados = pd.concat(partes, axis=0).sample(frac=1, random_state=2026).reset_index(drop=True)
    return datos_balanceados


def f_particionar_datos(datos: pd.DataFrame, variable_objetivo: str, proporcion_entrenamiento: float = 0.7) -> dict:
    """
    f_particionar_datos()
    Objetivo:
      Particionar en entrenamiento (70%) y validación (30%) con semilla 2026.
    Nota:
      Se usa estratificación por clase para mantener proporciones.
    """
    X = datos.drop(columns=[variable_objetivo])
    y = datos[variable_objetivo]

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y,
        train_size=proporcion_entrenamiento,
        random_state=2026,
        stratify=y
    )
    return {
        "datos_entrenamiento": pd.concat([X_train, y_train], axis=1),
        "datos_validacion": pd.concat([X_valid, y_valid], axis=1),
    }



## Cargar datos


In [3]:
ruta = "https://raw.githubusercontent.com/rpizarrog/machine_learning_r_python_casos_de_estudio/refs/heads/main/datos/online%20vs%20store%20shopping%20dataset.csv"
datos = f_cargar_datos(ruta)
datos.shape

(11789, 25)

# Visualizar los datos

Solo se muestran los primeros y últimos registros así como las primeras y últimas columnas del conjunto de datos.

In [15]:
f_visualizar_head_tail_reducido(datos, n=10)

,age,monthly_income,daily_internet_hours,smartphone_usage_years,gender,city_tier,shopping_preference
0,56,221111,6.5,12,Other,Tier 3,Store
1,69,96029,8.2,13,Male,Tier 3,Hybrid
2,46,19055,6.4,4,Female,Tier 3,Store
3,32,53170,6.4,11,Female,Tier 1,Store
4,60,244016,6.0,5,Male,Tier 3,Store
5,25,114976,7.6,6,Other,Tier 2,Store
6,78,43251,4.9,6,Other,Tier 2,Hybrid
7,38,150604,3.1,10,Other,Tier 2,Store
8,56,150996,7.3,2,Female,Tier 3,Store
9,75,63037,6.8,4,Female,Tier 3,Store


## Describir datos

Se despliega la estructura del conjunto de datos, destacando que *Python* de manera natural identifica a los atributos *gender*, *city_tier* y *shopping_preference* como objetos, es decir categóricos, lo cual no hace necesario una conversión.

Se muestran los estadísticos de los datos numéricos y categóricos.

In [5]:
resumen = f_describir_datos(datos)
print(resumen["structure"])
print(resumen["summary"])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11789 entries, 0 to 11788
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age                          11789 non-null  int64  
 1   monthly_income               11789 non-null  int64  
 2   daily_internet_hours         11789 non-null  float64
 3   smartphone_usage_years       11789 non-null  int64  
 4   social_media_hours           11789 non-null  float64
 5   online_payment_trust_score   11789 non-null  int64  
 6   tech_savvy_score             11789 non-null  int64  
 7   monthly_online_orders        11789 non-null  int64  
 8   monthly_store_visits         11789 non-null  int64  
 9   avg_online_spend             11789 non-null  int64  
 10  avg_store_spend              11789 non-null  int64  
 11  discount_sensitivity         11789 non-null  int64  
 12  return_frequency             11789 non-null  int64  
 13  avg_delivery_day

## Transformar datos

Del retorno de la función *f_transformar_factor()*, solo se muestran las frecuencias de los valores categóricos notándose el desbalance de las clases en la frecuencia.


In [6]:
datos = f_transformar_factor(datos)
resumen_transformado = f_describir_datos(datos)

# print(resumen_transformado["structure"])
# Solo columnas tipo factor esperadas (gender, city_tier, shopping_preference)
datos[["gender", "city_tier", "shopping_preference"]].describe(include="all")

,gender,city_tier,shopping_preference
count,11789,11789,11789
unique,3,3,3
top,Male,Tier 1,Store
freq,3966,3982,10244


## Unsampling

Extraer una muestra equilibrada las clases de acuerdo a la variable dependiente *shopping_preference* ejecutando la función *f_balancear_clases_undersamplig()*.


In [16]:
datos_balanceados = f_balancear_clases_undersamplig(datos, "shopping_preference")
datos_balanceados["shopping_preference"].value_counts()

/tmp/ipykernel_166/3411322993.py:90: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for clase, dfc in datos.groupby(variable_objetivo):


,count
shopping_preference,
Hybrid,369
Online,369
Store,369


# Desarrollo

Particionar los datos, *70%* para datos de entrenamiento y *30%* para datos de validación con semilla de aleatoriedad con valor de *2026*.

In [8]:
particion = f_particionar_datos(datos_balanceados, variable_objetivo="shopping_preference", proporcion_entrenamiento=0.7)

datos_entrenamiento = particion["datos_entrenamiento"]
datos_validacion = particion["datos_validacion"]

datos_entrenamiento.shape, datos_validacion.shape

((774, 25), (333, 25))

## Visualizar datos de entrenamiento

Solo se muestran los primeros y últimos seis registros de los datos de entrenamiento y las primeras y últimas columnas, al igual en los datos de validación.

In [9]:
f_visualizar_head_tail_reducido(datos_entrenamiento, n=6)

,age,monthly_income,daily_internet_hours,smartphone_usage_years,gender,city_tier,shopping_preference
0,28,119180,5.5,6,Other,Tier 1,Hybrid
1,44,18599,4.8,3,Other,Tier 3,Store
2,58,191554,9.1,9,Other,Tier 3,Store
3,47,189556,6.0,10,Female,Tier 2,Online
4,45,152892,5.6,14,Male,Tier 3,Hybrid
5,22,229513,10.5,2,Male,Tier 2,Store
6,...,...,...,...,...,...,...
7,48,96673,5.0,14,Male,Tier 3,Hybrid
8,39,42584,6.9,9,Other,Tier 3,Store
9,28,249566,4.3,9,Female,Tier 3,Online


## Visualiar datos de validación



In [10]:
f_visualizar_head_tail_reducido(datos_validacion, n=6)

,age,monthly_income,daily_internet_hours,smartphone_usage_years,gender,city_tier,shopping_preference
0,54,234802,6.0,9,Male,Tier 2,Online
1,19,101755,7.4,3,Other,Tier 3,Store
2,21,217558,3.9,10,Male,Tier 2,Store
3,59,22504,6.1,7,Other,Tier 2,Online
4,70,178944,7.3,14,Other,Tier 3,Online
5,41,55878,2.0,7,Female,Tier 2,Online
6,...,...,...,...,...,...,...
7,41,139726,6.6,6,Female,Tier 3,Store
8,18,202378,8.1,1,Other,Tier 2,Online
9,42,115153,7.8,5,Other,Tier 1,Hybrid


## Se construye el modelo

Para construir el modelo de regresión logística multinomial, se manda llamar la función *f_implementar_modelo_RL_multinomial()* previamente codificada para tal propósito; la variable dependiente es *shopping_preference* y el resto como variables independientes.


In [11]:
def f_check_numeric_df(X):
    """
    f_check_numeric_df()
    Objetivo:
      Diagnosticar columnas problemáticas para statsmodels:
      - columnas tipo object/category
      - valores no numéricos en columnas que deberían ser numéricas
      - NaN/Inf

    Acepta:
      DataFrame o Series (la convierte a DataFrame)
    Retorna:
      dict con hallazgos y además imprime un resumen útil.
    """
    # Convertir Series -> DataFrame
    if isinstance(X, pd.Series):
        X = X.to_frame()

    if not isinstance(X, pd.DataFrame):
        raise ValueError("X debe ser pandas DataFrame o Series.")

    # Tipos
    dtypes = X.dtypes

    # Columnas object/category
    cols_obj = X.select_dtypes(include=["object"]).columns.tolist()
    cols_cat = X.select_dtypes(include=["category"]).columns.tolist()

    # NaN por columna
    nan_cols = X.columns[X.isna().any()].tolist()

    # Inf por columna (solo en numéricas)
    cols_num = X.select_dtypes(include=["number"]).columns.tolist()
    inf_cols = []
    if cols_num:
        inf_cols = [c for c in cols_num if np.isinf(X[c].to_numpy()).any()]

    # Columnas "mixtas": object pero parecen numéricas (tienen strings)
    sospechosas = []
    ejemplos = {}
    for c in cols_obj:
        # tomar 10 valores únicos (no nulos) para inspección
        vals = X[c].dropna().astype(str).unique()[:10]
        ejemplos[c] = vals
        # si muchos parecen números, marcar como sospechosa
        try:
            pd.to_numeric(X[c].dropna().astype(str), errors="raise")
            sospechosas.append(c)
        except Exception:
            pass

    print("=== Diagnóstico de X ===")
    print(f"Dimensiones: {X.shape}")
    print("\nColumnas object:", cols_obj)
    print("Columnas category:", cols_cat)
    print("Columnas con NaN:", nan_cols)
    print("Columnas numéricas con Inf:", inf_cols)

    if cols_obj:
        print("\nEjemplos (primeros valores únicos) de columnas object:")
        for c in cols_obj:
            print(f" - {c}: {ejemplos[c]}")

    if sospechosas:
        print("\n Columnas object que parecen numéricas (conviene convertir con pd.to_numeric):")
        print(sospechosas)

    return {
        "cols_object": cols_obj,
        "cols_category": cols_cat,
        "cols_nan": nan_cols,
        "cols_inf": inf_cols,
        "sospechosas_numeric_as_object": sospechosas
    }

def f_implementar_modelo_RL_multinomial(
    datos_entrenamiento: pd.DataFrame,
    datos_validacion: pd.DataFrame,
    variable_dependiente: str,
    variables_independientes: list | None = None,
    drop_first_dummies: bool = True,
    alpha: float = 1.0,
    maxiter: int = 500,
    method: str = "l1",
    semilla: int = 2026
) -> dict:
    #------------------------------------------------------------
    # f_implementar_modelo_RL_multinomial()
    #
    # Objetivo:
    #   Construir un modelo de Regresión Logística Multinomial en Python
    #   usando statsmodels (MNLogit) con regularización (fit_regularized)
    #   para evitar problemas como separación perfecta/overflow.
    #
    # Argumentos:
    #   datos_entrenamiento   : DataFrame con datos de entrenamiento.
    #   datos_validacion      : DataFrame con datos de validación (para alinear X).
    #   variable_dependiente  : Nombre (str) de la variable objetivo (p.ej. "shopping_preference").
    #   variables_independientes : Lista de predictores. Si None, usa todas excepto la dependiente.
    #   drop_first_dummies    : Elimina la primera categoría al crear dummies (evita colinealidad).
    #   alpha                 : Fuerza de regularización (mayor = más penalización).
    #   maxiter               : Iteraciones máximas.
    #   method                : Tipo de regularización ("l1" o "l2").
    #   semilla               : Semilla para reproducibilidad (por defecto 2026).
    #
    # Retorna:
    #   dict con:
    #     - 'modelo'     : objeto MNLogit (no entrenado)
    #     - 'resultado'  : resultado entrenado (fit_regularized)
    #     - 'clases'     : lista de clases en el orden del modelo (clase base = clases[0])
    #     - 'scaler'     : StandardScaler ajustado en entrenamiento
    #     - 'X_train_s'  : matriz X de entrenamiento (escalada + const)
    #     - 'X_valid_s'  : matriz X de validación (escalada + const, alineada)
    #     - 'y_codes'    : códigos enteros de y (0..K-1)
    #     - 'X_cols'     : nombres de columnas usadas en X (útil para predicción consistente)
    #
    # Uso:
    #   res = f_implementar_modelo_RL_multinomial(datos_entrenamiento, datos_validacion,
    #                                            "shopping_preference")
    #   res["resultado"].params
    #------------------------------------------------------------

    np.random.seed(semilla)

    #-------------------------
    # 0) Validaciones
    #-------------------------
    if not isinstance(datos_entrenamiento, pd.DataFrame):
        raise ValueError("datos_entrenamiento debe ser un pandas.DataFrame.")
    if not isinstance(datos_validacion, pd.DataFrame):
        raise ValueError("datos_validacion debe ser un pandas.DataFrame.")
    if variable_dependiente not in datos_entrenamiento.columns:
        raise ValueError(f"No existe '{variable_dependiente}' en datos_entrenamiento.")
    if variable_dependiente not in datos_validacion.columns:
        raise ValueError(f"No existe '{variable_dependiente}' en datos_validacion.")

    # Si no se especifican independientes, usar todas menos la dependiente
    if variables_independientes is None:
        variables_independientes = [
            c for c in datos_entrenamiento.columns if c != variable_dependiente
        ]

    faltantes_train = [c for c in variables_independientes if c not in datos_entrenamiento.columns]
    faltantes_valid = [c for c in variables_independientes if c not in datos_validacion.columns]
    if faltantes_train:
        raise ValueError(f"Predictoras faltantes en entrenamiento: {faltantes_train}")
    if faltantes_valid:
        raise ValueError(f"Predictoras faltantes en validación: {faltantes_valid}")

    #-------------------------
    # 1) y -> category -> codes
    #-------------------------
    y_train_cat = datos_entrenamiento[variable_dependiente].astype("category")
    y_codes = y_train_cat.cat.codes.astype(int)
    clases = list(y_train_cat.cat.categories)

    if len(clases) < 3:
        raise ValueError("La variable dependiente debe tener al menos 3 clases (multinomial).")

    #-------------------------
    # 2) X raw (desde train y valid)
    #-------------------------
    X_train_raw = datos_entrenamiento[variables_independientes].copy()
    X_valid_raw = datos_validacion[variables_independientes].copy()

    #-------------------------
    # 3) Dummies
    #-------------------------
    X_train = pd.get_dummies(X_train_raw, drop_first=drop_first_dummies)
    X_valid = pd.get_dummies(X_valid_raw, drop_first=drop_first_dummies)

    #-------------------------
    # 4) Alinear columnas (valid a train)
    #-------------------------
    X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)

    #-------------------------
    # 5) Forzar float
    #-------------------------
    X_train = X_train.astype(float)
    X_valid = X_valid.astype(float)

    #-------------------------
    # 6) Escalar (para estabilidad numérica)
    #-------------------------
    scaler = StandardScaler()
    X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_valid_s = pd.DataFrame(scaler.transform(X_valid), columns=X_train.columns)

    #-------------------------
    # 7) Agregar constante UNA sola vez
    #-------------------------
    X_train_s = sm.add_constant(X_train_s, has_constant="add")
    X_valid_s = sm.add_constant(X_valid_s, has_constant="add")

    #-------------------------
    # 8) Quitar columnas duplicadas (seguridad)
    #-------------------------
    X_train_s = X_train_s.loc[:, ~X_train_s.columns.duplicated()].copy()
    X_valid_s = X_valid_s.loc[:, ~X_valid_s.columns.duplicated()].copy()

    #-------------------------
    # 9) Validación de finitud
    #-------------------------
    if not np.isfinite(X_train_s.to_numpy()).all():
        raise ValueError("X_train_s contiene NaN/Inf. Revisa datos y transformaciones.")
    if not np.isfinite(X_valid_s.to_numpy()).all():
        raise ValueError("X_valid_s contiene NaN/Inf. Revisa datos y transformaciones.")

    #-------------------------
    # 10) Alinear índices (CRÍTICO para statsmodels)
    #-------------------------
    X_train_s = X_train_s.reset_index(drop=True)
    y_codes = y_codes.reset_index(drop=True)

    #-------------------------
    # 11) Construir y entrenar MNLogit con regularización
    #-------------------------
    modelo = sm.MNLogit(y_codes, X_train_s)

    resultado = modelo.fit_regularized(
        method=method,   # "l1" o "l2"
        alpha=alpha,
        maxiter=maxiter,
        disp=False
    )

    return {
        "modelo": modelo,
        "resultado": resultado,
        "clases": clases,
        "scaler": scaler,
        "X_train_s": X_train_s,
        "X_valid_s": X_valid_s,
        "y_codes": y_codes,
        "X_cols": list(X_train_s.columns),
        "variable_dependiente": variable_dependiente,
        "variables_independientes": variables_independientes,
        "config": {
            "drop_first_dummies": drop_first_dummies,
            "alpha": alpha,
            "maxiter": maxiter,
            "method": method,
            "semilla": semilla
        }
    }

def f_evaluar_modelo(
    resultado_modelo: dict,
    datos_validacion: pd.DataFrame,
    umbral_accuracy: float = 0.70,
    fill_na: float = 0.0
) -> dict:
    #------------------------------------------------------------
    # f_evaluar_modelo()
    #
    # Objetivo:
    #   Evaluar un modelo de clasificación multinomial entrenado con
    #   statsmodels MNLogit (fit_regularized), usando datos de validación.
    #   Calcula matriz de confusión y métricas: Accuracy, Precision, Recall,
    #   F1 (por clase), además de Macro-F1 y decisión ACEPTAR/NO ACEPTAR.
    #
    # Argumentos:
    #   resultado_modelo : dict retornado por f_implementar_modelo_RL_multinomial()
    #                     (debe incluir: 'resultado', 'clases', 'scaler',
    #                      'variables_independientes', 'variable_dependiente',
    #                      y opcionalmente 'config').
    #   datos_validacion : DataFrame con predictores + variable dependiente.
    #   umbral_accuracy  : Umbral mínimo de exactitud para aceptar el modelo.
    #   fill_na          : Valor con el que se reemplazan NaN/Inf tras transformar.
    #
    # Retorna:
    #   dict con:
    #     - 'matriz_confusion'     : DataFrame (conteos)
    #     - 'accuracy'             : float
    #     - 'precision_por_clase'  : Series
    #     - 'recall_por_clase'     : Series
    #     - 'f1_por_clase'         : Series
    #     - 'macro_f1'             : float
    #     - 'decision'             : str ("ACEPTAR"/"NO ACEPTAR")
    #     - 'y_true'               : Series (clase real)
    #     - 'y_pred'               : Series (clase predicha)
    #     - 'probs'                : ndarray (probabilidades por clase)
    #
    # Requiere:
    #   numpy, pandas, sklearn.metrics.confusion_matrix
    #------------------------------------------------------------

    #-------------------------
    # 0) Validaciones
    #-------------------------
    for k in ["resultado", "clases", "scaler", "variables_independientes", "variable_dependiente"]:
        if k not in resultado_modelo:
            raise ValueError(f"resultado_modelo no contiene la llave requerida: '{k}'")

    res = resultado_modelo["resultado"]          # objeto fit_regularized
    clases = resultado_modelo["clases"]          # lista de clases (orden)
    scaler = resultado_modelo["scaler"]          # StandardScaler ajustado
    vars_ind = resultado_modelo["variables_independientes"]
    var_dep = resultado_modelo["variable_dependiente"]

    if var_dep not in datos_validacion.columns:
        raise ValueError(f"En datos_validacion no existe la variable dependiente '{var_dep}'")
    falt = [c for c in vars_ind if c not in datos_validacion.columns]
    if falt:
        raise ValueError(f"Faltan predictores en datos_validacion: {falt}")

    # y verdadero
    y_true = datos_validacion[var_dep].astype(str)

    #-------------------------
    # 1) Preparar X_valid: dummies + alineación + escalado + const
    #    (debe ser consistente con entrenamiento)
    #-------------------------
    X_valid_raw = datos_validacion[vars_ind].copy()

    # Crear dummies (mismo criterio)
    drop_first = True
    if "config" in resultado_modelo and "drop_first_dummies" in resultado_modelo["config"]:
        drop_first = bool(resultado_modelo["config"]["drop_first_dummies"])

    X_valid = pd.get_dummies(X_valid_raw, drop_first=drop_first)

    # Alinear columnas con entrenamiento (sin 'const' todavía)
    # X_cols incluye const; para alinear dummies, usamos columnas sin const
    X_cols = resultado_modelo.get("X_cols", None)
    if X_cols is None:
        raise ValueError("resultado_modelo debe incluir 'X_cols' (columnas usadas en entrenamiento).")

    cols_sin_const = [c for c in X_cols if c != "const"]
    X_valid = X_valid.reindex(columns=cols_sin_const, fill_value=0)

    # Forzar float
    X_valid = X_valid.astype(float)

    # Escalar con scaler entrenado
    X_valid_s = pd.DataFrame(scaler.transform(X_valid), columns=cols_sin_const)

    # Agregar constante y alinear exactamente a X_cols
    import statsmodels.api as sm
    X_valid_s = sm.add_constant(X_valid_s, has_constant="add")
    X_valid_s = X_valid_s.reindex(columns=X_cols, fill_value=0.0)

    # Limpiar NaN/Inf (por seguridad)
    X_valid_s = X_valid_s.replace([np.inf, -np.inf], np.nan).fillna(fill_na).astype(float)

    #-------------------------
    # 2) Predicción de probabilidades y clases
    #-------------------------
    probs = np.asarray(res.predict(X_valid_s))  # normalmente (n, K-1)

    K = len(clases)
    if probs.ndim == 1:
        # Caso raro: convertir a 2D
        probs = probs.reshape(-1, 1)

    # Reconstruir probabilidad de la clase base si el resultado trae K-1 columnas
    if probs.shape[1] == (K - 1):
        p_base = 1 - probs.sum(axis=1, keepdims=True)
        probs = np.hstack([p_base, probs])

    # Si por alguna razón trae K columnas, seguimos normal
    if probs.shape[1] != K:
        raise ValueError(f"Dimensión inesperada de probs: {probs.shape}. Se esperaban K={K} columnas.")

    idx = np.argmax(probs, axis=1)
    y_pred = pd.Series([clases[i] for i in idx], index=y_true.index)

    #-------------------------
    # 3) Matriz de confusión y métricas
    #-------------------------
    cm = confusion_matrix(y_true, y_pred, labels=clases)
    cm_df = pd.DataFrame(cm, index=clases, columns=clases)
    cm_df.index.name = "Real"
    cm_df.columns.name = "Predicho"

    total = cm.sum()
    accuracy = (np.trace(cm) / total) if total > 0 else 0.0

    # Recall por clase = TP / (TP+FN) = diag / row sum
    row_sums = cm.sum(axis=1)
    recall = np.divide(np.diag(cm), row_sums, out=np.zeros_like(row_sums, dtype=float), where=row_sums != 0)

    # Precision por clase = TP / (TP+FP) = diag / col sum
    col_sums = cm.sum(axis=0)
    precision = np.divide(np.diag(cm), col_sums, out=np.zeros_like(col_sums, dtype=float), where=col_sums != 0)

    # F1 por clase
    denom = (precision + recall)
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(denom, dtype=float), where=denom != 0)

    precision_s = pd.Series(precision, index=clases, name="precision")
    recall_s = pd.Series(recall, index=clases, name="recall")
    f1_s = pd.Series(f1, index=clases, name="f1")

    macro_f1 = float(f1_s.mean())

    # Decisión
    decision = "ACEPTAR" if accuracy >= umbral_accuracy else "NO ACEPTAR"

    return {
        "matriz_confusion": cm_df,
        "accuracy": float(accuracy),
        "precision_por_clase": precision_s,
        "recall_por_clase": recall_s,
        "f1_por_clase": f1_s,
        "macro_f1": macro_f1,
        "decision": decision,
        "y_true": y_true,
        "y_pred": y_pred,
        "probs": probs
    }




Se verifican las columnas ['gender', 'city_tier'] como parte de las variables independiente que son objetos para poder tratarlos y convertirlos en variables dummis para que el modelo en *Python* pueda funcionar.


In [12]:
# 1) define dependiente e independientes (igual que antes)
v_dependiente = "shopping_preference"
v_independientes = [c for c in datos_entrenamiento.columns if c != v_dependiente]

# 2) diagnóstico previo (opcional pero recomendable)
X_raw = datos_entrenamiento[v_independientes]
f_check_numeric_df(X_raw)   # te mostrará columnas problemáticas



=== Diagnóstico de X ===
Dimensiones: (774, 24)

Columnas object: []
Columnas category: ['gender', 'city_tier']
Columnas con NaN: []
Columnas numéricas con Inf: []


{'cols_object': [],
 'cols_category': ['gender', 'city_tier'],
 'cols_nan': [],
 'cols_inf': [],
 'sospechosas_numeric_as_object': []}

## Aplicar el modelo

Se manda llamar la función *f_implementar_modelo_RL_multinomial()* que construye el modelo de regresión logística y asegura que las variables independientes que son de tipo objeto se transformen a variables *dummis* que son valores numéricos.

Se muestran los coeficientes del modelo que pueden ser comparados contra los que arroja el caso construido en programación R y que sirven para hacer predicciones con datos de validación y evaluar el rendimiento, así como nuevas estimaciones con datos nuevos.


In [19]:
# Construir el modelo
# Definir variable dependiente
v_dependiente = "shopping_preference"

# Definir variables independientes
v_independientes = [
    c for c in datos_entrenamiento.columns
    if c != v_dependiente
]

# Construir modelo
resultado = f_implementar_modelo_RL_multinomial(
    datos_entrenamiento = datos_entrenamiento,
    datos_validacion = datos_validacion,
    variable_dependiente = v_dependiente,
    variables_independientes = v_independientes,
    alpha = 1.0,
    maxiter = 500
)


# Coeficientes del modelo
np.round(resultado["resultado"].params, 4)

,0,1
const,-15.6146,2.6445
age,0.1309,0.0000
monthly_income,0.0000,0.1596
daily_internet_hours,1.7945,-0.6823
smartphone_usage_years,0.0000,-0.0365
social_media_hours,-0.0396,-0.2051
online_payment_trust_score,2.0504,-0.9122
tech_savvy_score,3.5395,-1.8047
monthly_online_orders,2.9053,-1.4854
monthly_store_visits,-2.4874,0.8233


## Evaluar el modelo

Ahora llamando la función *f_evaluar_modelo()* se puede evaluar el modelo interesando por el momento el valor de *accuracy* o exactitud general del modelo, buscando que este valor esté por encima del *70%* para aceptarlo como modelo eficiente predicitivo de clasificación.

En la variable *evaluacion* está lo que retorna de la función y están los estadísticos que permiten precisamente analizar el comportamiento del modelo.





In [14]:
evaluacion = f_evaluar_modelo(
    resultado_modelo = resultado,
    datos_validacion = datos_validacion,
    umbral_accuracy = 0.70
)

# Matriz de confusión
print("Matriz de confusión")
print(evaluacion["matriz_confusion"])

# Sólo el estadístico accuracy
print("\n")
print("Exactitud del modelo accuracy")
print(evaluacion["accuracy"])

# Se acepta o se rechaza el modelo
print("\n")
print("Se acepta o se rechaza el modelo por encima del 70%")
print(evaluacion["decision"])

Matriz de confusión
Predicho  Hybrid  Online  Store
Real                           
Hybrid       109       1      1
Online        12      99      0
Store          6       0    105


Exactitud del modelo accuracy
0.93993993993994


Se acepta o se rechaza el modelo por encima del 70%
ACEPTAR


# Interpreración del caso

El caso describe la implementación y evaluación de un modelo predictivo de clasificación de regresión logística multinomial.

El contexto de los datos son características de comportamiento de compradores y la predicción sería que tipo de compra va a realziar un cliente si será en línea, presencial o en modo híbrido.

El caso siguió la metodología sugerida desde contextualizar los datos, definir el objetivo, describir y transformar los datos, implementar y evaluar el modelo; así como concluir con la interpretación del mismo.

En *Python* fue necesario hacer algunos ajustes en los datos antes de implementar el modelo, se hizo una transformación importante:

Primero, se modificaron variables categóricas como *'gender'* y *'city_tier'* a variable *dummis* que implica convertirlas a valores numéricos en lugar de categóricos, técnicamente sería modificar variables categóricas (cualitativas) en un conjunto de variables numéricas binarias (0 y 1) o más valores numéricos dependiente de la cantidad de clases.

Segundo, se escalaron los datos numéricos como requisito en las variables independientes, además de usar una función robusta para limpiar los datos antes de usarlos en el modelo.

Fue necesario cargar las librerías adecuadas y crear las funciones necesarias para la correcta ejecución del caso. Las funciones fueron soportadas verificadas por inteligencia artificial generativa específicamente *CITA*.

Se cargaron los mismos datos provistos del enlace "https://raw.githubusercontent.com/rpizarrog/machine_learning_r_python_casos_de_estudio/refs/heads/main/datos/online%20vs%20store%20shopping%20dataset.csv" y se hizo una partición de datos en 70% datos de entrenamiento y 30% datos de validación.

La evaluación del modelo de regresión logística para estos datos resultó que tiene una exactitud del *93%*, muy similar al *94%* que arroja el caso en *R*, por lo cual se acepta el modelo ya que se definió un criterio que este estadístico estuviera por encima del *70%*.

En caso completo se encuentra en el enlace siguiente:https://github.com/rpizarrog/machine_learning_r_python_casos_de_estudio/blob/main/notebook_Python/Caso_ejemplo_general_con_datos_de_comportamienos_de_compradores.ipynb  


